In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier


In [3]:
print("Working on Titanic dataset")
data = pd.read_csv("titanic.csv")
data.info()
print(data.isnull().sum())

Working on Titanic dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked

In [7]:
def fill_missing_ages(df: pd.DataFrame) -> pd.DataFrame:
    """
    filling missing ages in dataFrame (df)
    """
    age_fill_map = {}

    for pclass in df["Pclass"].unique():
        if pclass not in age_fill_map:
            age_fill_map[pclass] = df[df["Pclass"] == pclass]["Age"].median()

    # Apply the median onto df if row["Age"] is null otherwize keep the original age
    df["Age"] = df.apply(lambda row: age_fill_map[row["Pclass"]] if pd.isnull(row["Age"]) else row["Age"], axis=1)
    # df["Age"].fillna(df["Pclass"].map(age_fill_map), inplace=True)
    print(f"Age fill map: {age_fill_map}")

    return df

In [8]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop unused columns, fill null values and convert in number type
    """
    df.drop(columns=["PassengerId","Name","Ticket","Cabin"], inplace=True)

    # Fill the missing values as "S" for Southampton, the most common embarkation point in the data
    df["Embarked"].fillna("S", inplace=True)
    df.drop(columns=["Embarked"], inplace=True)

    fill_missing_ages(df)

    # Convert Gender for model
    df["Sex"] = df["Sex"].map({'male': 1, 'female': 0})

    # Feature engineering
    df["FamilySize"] = df["SibSp"] + df["Parch"] # parents + children
    df["IsAlone"] = np.where(df["FamilySize"] == 0, 1, 0) # where there is no one then insert 1
    df["FareBin"] = pd.qcut(df["Fare"], 4, labels=False) # categorization for ticket prices
    df["AgeBin"] = pd.cut(df["Age"], bins=[0,12,20,40,60, np.inf], labels=False) # bins for ranged age of passengers
    print(df)
    with open("data_preprocessed.csv", "w") as f:
        df.to_csv(f, index=False)

    return df

In [9]:
# Preprocessing data
print("Preprocessing data...")
preprocessed_data = preprocess_data(data)

# Create Features / Target Variables (Make Flashcards)
X = preprocessed_data.drop(columns=["Survived"])
y = preprocessed_data["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

Preprocessing data...
Age fill map: {np.int64(3): np.float64(24.0), np.int64(2): np.float64(26.5), np.int64(1): np.float64(42.0)}
     Survived  Pclass  Sex   Age  SibSp  Parch      Fare  FamilySize  IsAlone  \
0           0       3    1  34.5      0      0    7.8292           0        1   
1           1       3    0  47.0      1      0    7.0000           1        0   
2           0       2    1  62.0      0      0    9.6875           0        1   
3           0       3    1  27.0      0      0    8.6625           0        1   
4           1       3    0  22.0      1      1   12.2875           2        0   
..        ...     ...  ...   ...    ...    ...       ...         ...      ...   
413         0       3    1  24.0      0      0    8.0500           0        1   
414         1       1    0  39.0      0      0  108.9000           0        1   
415         0       3    1  38.5      0      0    7.2500           0        1   
416         0       3    1  24.0      0      0    8.0500    

/var/folders/42/p19smyt94wn_jpbmnlws7tgr0000gn/T/ipykernel_47768/4036095894.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Embarked"].fillna("S", inplace=True)
